In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col, trim, upper, lower

#SparkSession limpa
spark = SparkSession.builder \
    .appName("Bronze-to-Silver-Olist") \
    .getOrCreate()
    
input_path = "s3a://bronze/olist/customers"
output_path = "s3a//silver/olist/customers"

print(f"Lendo dados da camada Bronze: {input_path}")

Lendo dados da camada Bronze: s3a://bronze/olist/customers


In [16]:
# Lendo os dados de Bronze
df_bronze = spark.read.parquet(input_path)

df_bronze.show(5, truncate=False)
df_bronze.printSchema()

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+-------------------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|data_processamento_bronze|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+-------------------------+
|f2a1d75b74d9ec748af88e894cd87597|15ee900ec703c9a10082b12bcbdcb307|68590                   |jacunda              |PA            |2026-06-13 05:40:34.82952|
|f15272fe9d0e2ae3297185f18d3bac46|11e74a9cbe1158d1c934d4c166f59ff3|15056                   |sao jose do rio preto|SP            |2026-06-13 05:40:34.82952|
|7324ecb0ff143f561193d22bea7d63fb|c6be127fa6e30c6f705a205236a1f310|13302                   |itu                  |SP            |2026-06-13 05:40:34.82952|
|7accf3d920f47c07f5bfbc88f53f9926|a7f1a6dc9ba06844bd84b810efa56d

In [ ]:
#Tratamento dos dados
df_silver = df_bronze \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("customer_unique_id", trim(col("customer_unique_id"))) \
    .withColumn("customer_zip_code_prefix", trim(col("customer_zip_code_prefix"))) \
    .withColumn("customer_city", lower(trim(col("customer_city")))) \
    .withColumn("customer_state", upper(trim(col("customer_state")))) \
    .withColumn("dt_criacao_silver", current_timestamp())

df_silver.show(5, truncate=False)

In [21]:
df_silver.write.mode("overwrite").parquet(output_path)
print("Dados gravados com sucesso na camada Silver!")

Dados gravados com sucesso na camada Silver!
